<a href="https://colab.research.google.com/github/Nutchayapon/Super_AI_Engineer-SS.6/blob/main/5D_Hackathon_Sleep_Stage_Classification_601661.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Super AI Engineer SS6] :  5 Domains Hackathon

#  Sleep Stage Classification

In [ ]:
import os, csv, re, time, requests
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import userdata

# Data

In [ ]:
KAGGLE_API_KEY = userdata.get('KAGGLE_API')
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_KEY
!kaggle competitions download -c super-ai-engineer-ss-6-sleep-stage-classification
!unzip -q super-ai-engineer-ss-6-sleep-stage-classification

Resuming from 20971520 bytes (2268853571 bytes left)...
100% 2.13G/2.13G [01:27<00:00, 26.0MB/s]



# Explore


In [ ]:
import os
os.listdir() # ดูว่ามีไฟล์ชื่ออะไรบ้าง

['.config',
 'super-ai-engineer-ss-6-sleep-stage-classification.zip',
 'train',
 'sample_submission.csv',
 '.ipynb_checkpoints',
 'test_segment',
 'sample_data']

In [ ]:
import os
from pathlib import Path

TRAIN_DIR = Path("train")
TEST_DIR  = Path("test_segment")

# ดู subjects ทั้งหมด
subjects = sorted(TRAIN_DIR.iterdir())
print(f"จำนวน subjects: {len(subjects)}")
print(f"ตัวอย่างชื่อ: {[s.name for s in subjects[:5]]}")

# ดูว่าในแต่ละ folder มีไฟล์อะไรบ้าง
first_subject = subjects[0]
print(f"\nไฟล์ใน {first_subject.name}/:")
for f in sorted(first_subject.iterdir()):
    print(f"  {f.name}")

จำนวน subjects: 1
ตัวอย่างชื่อ: ['train']

ไฟล์ใน train/:
  train001.csv
  train002.csv
  train003.csv
  train004.csv
  train005.csv
  train006.csv
  train007.csv
  train008.csv
  train009.csv
  train010.csv
  train011.csv
  train012.csv
  train013.csv
  train014.csv
  train015.csv
  train016.csv
  train017.csv
  train018.csv
  train019.csv
  train020.csv
  train021.csv
  train022.csv
  train023.csv
  train024.csv
  train025.csv
  train026.csv
  train027.csv
  train028.csv
  train029.csv
  train030.csv
  train031.csv


In [ ]:
first_subject = sorted(Path("train").iterdir())[0]

# ลองหาไฟล์ที่มีอยู่
files = list(first_subject.iterdir())
for f in files:
    df = pd.read_csv(f)
    print(f"=== {f.name} ===")
    print(f"Shape: {df.shape}")
    print(df.head(3))
    print()

=== train010.csv ===
Shape: (410400, 9)
          BVP     ACC_X      ACC_Y      ACC_Z       TEMP       EDA         HR  \
0 -150.564910 -4.948668 -39.541449  46.473564  34.599361  0.425784  62.941055   
1 -209.497936 -4.279075 -39.543475  46.902812  34.599377  0.425795  62.941110   
2 -116.582282 -4.490656 -39.540286  46.333927  34.599317  0.425762  62.941477   

        IBI Sleep_Stage  
0  0.834204           W  
1  0.815431           W  
2  0.800547           W  

=== train021.csv ===
Shape: (393120, 9)
        BVP      ACC_X      ACC_Y      ACC_Z       TEMP       EDA         HR  \
0  9.665813 -19.757577  37.555707  48.452465  29.191976  0.409083  62.605730   
1  8.257464 -20.438194  37.153789  48.440073  29.191968  0.409083  62.605440   
2  9.449820 -20.215478  37.618679  48.421970  29.191979  0.409084  62.603383   

       IBI Sleep_Stage  
0  1.08123           W  
1  1.08123           W  
2  1.08123           W  

=== train014.csv ===
Shape: (401280, 9)
          BVP      ACC_X    

In [ ]:
import pandas as pd
from pathlib import Path

# ดู test files
test_files = sorted(Path("test_segment").iterdir())
print(f"จำนวน test files: {len(test_files)}")
print(f"ตัวอย่างชื่อ: {[f.name for f in test_files[:5]]}")

# ดูหน้าตา test file
sample_test = pd.read_csv

จำนวน test files: 1
ตัวอย่างชื่อ: ['test_segment']


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# --- path---
TRAIN_DIR = Path("train/train")
TEST_DIR  = Path("test_segment/test_segment")
SAMPLE_SUB = Path("sample_submission.csv")

In [ ]:
def load_train_data(train_dir: Path):
    """
    โหลดไฟล์ train ทุกไฟล์ แล้วตัดเป็น epoch ขนาด 480 แถว
    คืนค่า X (features รอสกัด) และ y (labels)
    """
    all_epochs  = []  # เก็บ raw epoch แต่ละชิ้น
    all_labels  = []  # เก็บ label ของแต่ละ epoch

    FEATURE_COLS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z",
                    "TEMP", "EDA", "HR", "IBI"]
    LABEL_COL    = "Sleep_Stage"
    EPOCH_LEN    = 480  # 30 วิ × 16 Hz

    for csv_file in sorted(train_dir.glob("*.csv")):
        df = pd.read_csv(csv_file)

        n_epochs = len(df) // EPOCH_LEN  # กี่ epoch ใน subject นี้

        for i in range(n_epochs):
            epoch = df.iloc[i*EPOCH_LEN : (i+1)*EPOCH_LEN]

            signal = epoch[FEATURE_COLS].values  # shape (480, 8)

            # ใช้ label ของแถวแรกของ epoch นั้น
            # (label เดิมบอกทุก row ซ้ำๆ กันอยู่แล้ว)
            label = epoch[LABEL_COL].iloc[0]

            all_epochs.append(signal)
            all_labels.append(label)

        print(f"  {csv_file.name}: {n_epochs} epochs")

    return all_epochs, all_labels

print("กำลังโหลด train data...")
epochs, labels = load_train_data(TRAIN_DIR)
print(f"\nรวม: {len(epochs)} epochs, {len(set(labels))} classes")
print(f"Classes: {sorted(set(labels))}")

กำลังโหลด train data...
  train001.csv: 743 epochs
  train002.csv: 829 epochs
  train003.csv: 836 epochs
  train004.csv: 733 epochs
  train005.csv: 843 epochs
  train006.csv: 821 epochs
  train007.csv: 748 epochs
  train008.csv: 807 epochs
  train009.csv: 824 epochs
  train010.csv: 855 epochs
  train011.csv: 786 epochs
  train012.csv: 796 epochs
  train013.csv: 955 epochs
  train014.csv: 836 epochs
  train015.csv: 821 epochs
  train016.csv: 720 epochs
  train017.csv: 840 epochs
  train018.csv: 796 epochs
  train019.csv: 862 epochs
  train020.csv: 842 epochs
  train021.csv: 819 epochs
  train022.csv: 760 epochs
  train023.csv: 799 epochs
  train024.csv: 805 epochs
  train025.csv: 843 epochs
  train026.csv: 844 epochs
  train027.csv: 810 epochs
  train028.csv: 798 epochs
  train029.csv: 941 epochs
  train030.csv: 795 epochs
  train031.csv: 168 epochs

รวม: 24675 epochs, 5 classes
Classes: ['N1', 'N2', 'N3', 'R', 'W']


In [ ]:
def extract_features_v2(signal: np.ndarray) -> np.ndarray:
    features = []
    SR = 16  # sampling rate

    # ดึงแต่ละ channel
    bvp   = signal[:, 0]
    acc_x = signal[:, 1]
    acc_y = signal[:, 2]
    acc_z = signal[:, 3]
    temp  = signal[:, 4]
    eda   = signal[:, 5]
    hr    = signal[:, 6]
    ibi   = signal[:, 7]

    # =============================================
    # 1. features เดิม (time + frequency domain)
    # =============================================
    for ch in range(signal.shape[1]):
        x = signal[:, ch]
        features += [
            np.mean(x), np.std(x), np.min(x), np.max(x),
            np.max(x) - np.min(x),
            np.percentile(x, 25), np.percentile(x, 75),
            np.percentile(x, 75) - np.percentile(x, 25),
            np.mean(np.abs(x - np.mean(x))),
            np.sum(x**2) / len(x),
            pd.Series(x).skew(),
            pd.Series(x).kurt(),
        ]
        fft_vals = np.abs(np.fft.rfft(x))
        freqs    = np.fft.rfftfreq(len(x), d=1/SR)
        for lo, hi in [(0, 0.5), (0.5, 2.0), (2.0, 4.0), (4.0, 8.0)]:
            mask = (freqs >= lo) & (freqs < hi)
            features.append(np.sum(fft_vals[mask]**2))
        features.append(freqs[np.argmax(fft_vals)])

    # =============================================
    # 2. ACC magnitude — บอกว่าขยับมากแค่ไหน
    #    N3 = นิ่งมาก, W = ขยับเยอะ
    # =============================================
    acc_mag = np.sqrt(acc_x**2 + acc_y**2 + acc_z**2)
    features += [
        np.mean(acc_mag),
        np.std(acc_mag),
        np.max(acc_mag),
        np.sum(acc_mag > np.mean(acc_mag) + 2*np.std(acc_mag)),  # นับครั้งที่ขยับแรง
    ]

    # =============================================
    # 3. HRV (Heart Rate Variability) จาก IBI
    #    REM = HRV สูง, N3 = HRV ต่ำ, W = HRV ปานกลาง
    # =============================================
    ibi_clean = ibi[ibi > 0]  # กรอง 0 ออก
    if len(ibi_clean) > 1:
        ibi_diff = np.diff(ibi_clean)
        features += [
            np.std(ibi_clean),                          # SDNN
            np.sqrt(np.mean(ibi_diff**2)),              # RMSSD — บอก REM ได้ดี
            np.mean(np.abs(ibi_diff)),                  # mean absolute difference
            np.max(ibi_clean) - np.min(ibi_clean),      # range ของ IBI
        ]
    else:
        features += [0, 0, 0, 0]

    # =============================================
    # 4. EDA (เหงื่อ/ความเครียด)
    #    W/REM = EDA สูงกว่า N3
    # =============================================
    eda_diff = np.diff(eda)
    features += [
        np.mean(np.abs(eda_diff)),   # อัตราการเปลี่ยนแปลง EDA
        np.sum(eda_diff > 0),        # นับครั้งที่ EDA เพิ่ม
    ]

    # =============================================
    # 5. TEMP trend
    #    หลับลึก (N3) = อุณหภูมิลดลงช้าๆ
    # =============================================
    temp_trend = np.polyfit(np.arange(len(temp)), temp, 1)[0]  # slope
    features.append(temp_trend)

    # =============================================
    # 6. HR statistics เพิ่มเติม
    #    N3 = HR ต่ำและนิ่ง, REM = HR ผันผวน
    # =============================================
    hr_diff = np.diff(hr)
    features += [
        np.mean(np.abs(hr_diff)),    # HR variability
        np.std(hr_diff),
    ]

    return np.array(features)

# สกัด features ใหม่
print("สกัด features v2...")
X_train_v2 = np.array([extract_features_v2(ep) for ep in tqdm(epochs)])
print(f"X_train_v2 shape: {X_train_v2.shape}")

สกัด features v2...


100%|██████████| 24675/24675 [03:48<00:00, 108.01it/s]


X_train_v2 shape: (24675, 149)


# Train Model

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from tqdm import tqdm

# แปลง label เป็นตัวเลข
le = LabelEncoder()
y_train = le.fit_transform(labels)


# ── Model ──
model_xgb = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        eval_metric="mlogloss",
    ))
])

# ── CV ──
print("Cross-Validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_xgb = cross_val_score(
    model_xgb, X_train_v2, y_train,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1        # ← แสดง progress ของแต่ละ fold
)
print(f"\nCV Weighted F1: {scores_xgb.mean():.4f} ± {scores_xgb.std():.4f}")

# ── Train ──
print("\n Train...")
model_xgb.fit(X_train_v2, y_train)
print("✅ Train เสร็จแล้ว!")

Cross-Validation...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  9.2min finished



CV Weighted F1: 0.7658 ± 0.0057

 Train...
✅ Train เสร็จแล้ว!


In [ ]:
def predict_test_v2(test_dir, model, label_encoder):
    FEATURE_COLS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z",
                    "TEMP", "EDA", "HR", "IBI"]
    results = []
    test_files = sorted(test_dir.rglob("*.csv"))
    print(f"\nพบ test files: {len(test_files)} ไฟล์")

    for f in tqdm(test_files, desc="Predicting"):
        df    = pd.read_csv(f)
        epoch = df[FEATURE_COLS].values
        feat  = extract_features_v2(epoch).reshape(1, -1)
        pred  = label_encoder.inverse_transform(model_xgb.predict(feat))[0]
        results.append({"id": f.stem, "labels": pred})

    return pd.DataFrame(results)

submission_xgb = predict_test_v2(TEST_DIR, model_xgb, le)

print(f"\nDistribution:")
print(submission_xgb["labels"].value_counts())

# ── Save ──
sample_sub = pd.read_csv(SAMPLE_SUB)
submission_xgb = (submission_xgb
                  .set_index("id")
                  .loc[sample_sub["id"]]
                  .reset_index())
submission_xgb.to_csv("submission_xgb.csv", index=False)
print("\n✅ บันทึก submission_xgb.csv แล้ว!")
print(submission_xgb.head(10))


พบ test files: 7832 ไฟล์


Predicting: 100%|██████████| 7832/7832 [02:51<00:00, 45.60it/s]



Distribution:
labels
N2    5995
W     1721
N1      64
R       52
Name: count, dtype: int64

✅ บันทึก submission_xgb.csv แล้ว!
              id labels
0  test001_00000     N2
1  test001_00001     N2
2  test001_00002     N2
3  test001_00003     N2
4  test001_00004     N2
5  test001_00005     N2
6  test001_00006     N2
7  test001_00007     N2
8  test001_00008     N2
9  test001_00009      W
